In [1]:
"""
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
"""

'\n# Input data files are available in the read-only "../input/" directory\n# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n\nimport os\nfor dirname, _, filenames in os.walk(\'/kaggle/input\'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n\n# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" \n# You can also write temporary files to /kaggle/temp/, but they won\'t be saved outside of the current session\n'

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np   
import pandas as pd    

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline




El objetivo de esta practica es predecir la variable target.

La variable target es un valor continuo, por lo que usaremos un modelo de regresion, para este caso podriamos usar.

Modelos comunes:

- LinearRegression
- RandomForestRegressor
- XGBoostRegressor

Procedemos a cargar los archivos.


In [3]:
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

# 1.- Analisis de datos

In [4]:
print('Train set shape:', train.shape)
print('Test set shape:', test.shape)

Train set shape: (300000, 16)
Test set shape: (200000, 15)


In [5]:
train.head()

,id,cont1,cont2,cont3,cont4,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,target
0,1,0.670390,0.811300,0.643968,0.291791,0.284117,0.855953,0.890700,0.285542,0.558245,0.779418,0.921832,0.866772,0.878733,0.305411,7.243043
1,3,0.388053,0.621104,0.686102,0.501149,0.643790,0.449805,0.510824,0.580748,0.418335,0.432632,0.439872,0.434971,0.369957,0.369484,8.203331
2,4,0.834950,0.227436,0.301584,0.293408,0.606839,0.829175,0.506143,0.558771,0.587603,0.823312,0.567007,0.677708,0.882938,0.303047,7.776091
3,5,0.820708,0.160155,0.546887,0.726104,0.282444,0.785108,0.752758,0.823267,0.574466,0.580843,0.769594,0.818143,0.914281,0.279528,6.957716
4,8,0.935278,0.421235,0.303801,0.880214,0.665610,0.830131,0.487113,0.604157,0.874658,0.863427,0.983575,0.900464,0.935918,0.435772,7.951046


In [6]:
print('VALORES FALTANTES TRAIN:')
print(train.isna().sum())
print('')
print('VALORES FALTANTES TEST:')
print(test.isna().sum())

VALORES FALTANTES TRAIN:
id        0
cont1     0
cont2     0
cont3     0
cont4     0
cont5     0
cont6     0
cont7     0
cont8     0
cont9     0
cont10    0
cont11    0
cont12    0
cont13    0
cont14    0
target    0
dtype: int64

VALORES FALTANTES TEST:
id        0
cont1     0
cont2     0
cont3     0
cont4     0
cont5     0
cont6     0
cont7     0
cont8     0
cont9     0
cont10    0
cont11    0
cont12    0
cont13    0
cont14    0
dtype: int64


In [7]:
print(f'Duplicados en Train: {train.duplicated().sum()}, ({np.round(100*train.duplicated().sum()/len(train),1)}%)')
print('')
print(f'Duplicados en Test: {test.duplicated().sum()}, ({np.round(100*test.duplicated().sum()/len(test),1)}%)')

Duplicados en Train: 0, (0.0%)

Duplicados en Test: 0, (0.0%)


In [8]:
train.describe()

,id,cont1,cont2,cont3,cont4,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,target
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,249825.145857,0.506873,0.497898,0.521557,0.515683,0.502022,0.526515,0.487890,0.525163,0.459857,0.520532,0.483926,0.506877,0.553442,0.503713,7.905661
std,144476.732562,0.203976,0.228159,0.200770,0.233035,0.220701,0.217909,0.181096,0.216221,0.196685,0.201854,0.220082,0.218947,0.229730,0.208238,0.733071
min,1.000000,-0.082263,-0.031397,0.020967,0.152761,0.276377,0.066166,-0.097666,0.217260,-0.240604,-0.085046,0.083277,0.088635,0.029950,0.166367,0.000000
25%,124656.500000,0.343078,0.319170,0.344096,0.294935,0.284108,0.356163,0.346600,0.341486,0.330832,0.375465,0.300474,0.310166,0.350472,0.308673,7.329367
50%,249738.500000,0.484005,0.553209,0.551471,0.482880,0.451733,0.470988,0.466825,0.483460,0.416843,0.458877,0.441916,0.486599,0.487707,0.431845,7.940571
75%,374836.250000,0.643789,0.731263,0.648315,0.748705,0.670660,0.694043,0.581292,0.685250,0.575041,0.700292,0.679128,0.694453,0.768479,0.712653,8.470084
max,499999.000000,1.016227,0.859697,1.006955,1.010402,1.034261,1.043858,1.066167,1.024427,1.004114,1.199951,1.022620,1.049025,0.977845,0.868506,10.267569


Separaremos la variable objetivo "target" de los demas datos como entradas.

In [9]:
X = train.drop(columns=["target"])
y = train["target"]

# 2.- Entrenar el modelo base

In [10]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [13]:
y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.5907360332542297
MSE: 0.49852152687617673
RMSE: 0.7060605688439036
R2: 0.06522165834784899


In [14]:
y_mean = y_train.mean()
baseline_pred = np.full_like(y_val, y_mean)

print("Baseline R2:", r2_score(y_val, baseline_pred))

Baseline R2: -2.3935250658357887e-05


In [15]:
corr = train.corr()["target"].sort_values(ascending=False)
print(corr.head(16))

target    1.000000
cont7     0.067234
cont2     0.067102
cont3     0.058936
cont11    0.050996
cont12    0.047809
cont6     0.027955
cont8     0.014698
cont4     0.005522
id        0.001347
cont5    -0.005358
cont14   -0.006609
cont13   -0.006642
cont9    -0.013029
cont10   -0.021143
cont1    -0.032994
Name: target, dtype: float64


In [16]:
print("Train R2:", model.score(X_train, y_train)) 
print("Val R2:", model.score(X_val, y_val))

Train R2: 0.8691660691309044
Val R2: 0.06522165834784899


In [17]:
importances = model.feature_importances_ 
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False) 
print(feat_imp.head(16))

cont3     0.072437
cont2     0.071975
cont4     0.071707
cont13    0.070129
cont7     0.067833
cont1     0.067333
cont12    0.065912
cont10    0.065879
cont6     0.065277
cont8     0.065029
cont11    0.064811
cont14    0.064207
cont9     0.063820
cont5     0.063243
id        0.060409
dtype: float64


*Nota*

Como resultado vemos metricas muy pobres (MAE, RMSE, R2) tendremos que ajustar.

- No es preciso error ~8% (0.59 / 7.5)
- R2 es muy pobre solo 6.5% de prediccion.
- Baseline mejora muy poco solo un .00002
- La correlacion es muy pobre, teniendo como mejores cont7, cont2, cont3 y cont11.
- El modelo aprende bien los datos de entrenamiento a un 87%.
- Diferencia enorme entre train y validacion (0.065) = sobreajuste (overfitting).
- Importancias casi iguales ninguna descata.
- Esto nos indica que los datos son muy pobres.


*Prespectiva*

- Devemos eliminar el ID.
- Tenemos muchas lineas 300K buscaremos mejores modelos como HistGradientBoostingRegressor.
- Como no hay correlaciones fuertes, necesitas interacciones, crearemos algunos features.


In [18]:
X = X.drop(columns=["id"])

X["cont7_cont2"] = X["cont7"] * X["cont2"]
X["cont7_sq"] = X["cont7"] ** 2
X["cont2_sq"] = X["cont2"] ** 2

In [19]:

gbr = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)



In [20]:
gbr.fit(X_train, y_train)
print("Train R2:", gbr.score(X_train, y_train))
print("Val R2:", gbr.score(X_val, y_val))

Train R2: 0.1093664409129228
Val R2: 0.08039975136483013


In [21]:
y_pred = gbr.predict(X_val)

print("R2:", r2_score(y_val, y_pred))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("MSE:", mean_squared_error(y_val, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))

R2: 0.08039975136483013
MAE: 0.5877741905977347
MSE: 0.49042698106917754
RMSE: 0.700304920066379


*Nota*

- Mejoro R2 de 0.65 a 0.80
- Se mejoro el sobreajuste de train y Val

*Prespectiva*

- Ajustaremos parametros de Hist Gradient.
- Probar transformadas simples a los valores con mejor correlacion.
- Validar el gbr.score y ver si mejoro.


In [22]:
for c in ["cont2", "cont3", "cont7"]:
    X[c + "_sq"] = X[c] ** 2

In [23]:
gbr = HistGradientBoostingRegressor(
    max_depth=4,
    learning_rate=0.03,
    max_iter=600,
    l2_regularization=0.1,
    random_state=42
)

In [24]:
gbr.fit(X_train, y_train)
print("Train R2:", gbr.score(X_train, y_train))
print("Val R2:", gbr.score(X_val, y_val))

Train R2: 0.08924424553527388
Val R2: 0.0739120597098154


*Nota Final*

Vemos que el modelo anterior es mejor, Train y validation son mas carcanos en este segundo modelo esta mejor regulazirado, pero empeora nuestra prediccion este seria el limite.

# 3. Entrenar modelo final


In [25]:
X = train.drop(columns=["target"])
y = train["target"]

test_ids = test["id"]

In [26]:
def feature_engineering(df):
    df = df.copy()
    
    # eliminar id
    df = df.drop(columns=["id"])
    
    # features cuadráticas (ejemplo)
    df["cont2_sq"] = df["cont2"] ** 2
    df["cont3_sq"] = df["cont3"] ** 2
    df["cont7_sq"] = df["cont7"] ** 2
    
    return df

In [27]:
pipeline = Pipeline(steps=[
    ("features", FunctionTransformer(feature_engineering)),
    ("model", HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        random_state=42
    ))
])

In [28]:
pipeline.fit(X, y)


Pipeline(steps=[('features',
                 FunctionTransformer(func=<function feature_engineering at 0x0000021318BB5B20>)),
                ('model',
                 HistGradientBoostingRegressor(learning_rate=0.05, max_depth=6,
                                               max_iter=300,
                                               random_state=42))])

In [29]:
test_preds = pipeline.predict(test)

In [33]:
submission = pd.DataFrame({
    "Id": test_ids,
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)

In [34]:
print(submission.head())
print(submission.shape)
print(submission["target"].describe())

   Id    target
0   0  7.954907
1   2  7.854529
2   6  7.940974
3   7  8.198609
4  10  8.229402
(200000, 2)
count    200000.000000
mean          7.904157
std           0.189131
min           7.180737
25%           7.774992
50%           7.890761
75%           8.018688
max           9.294396
Name: target, dtype: float64
